<a href="https://colab.research.google.com/github/va4756/bigdata_raejung/blob/main/LLM_product_chap2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2-6 RNN과 LSTM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install gensim
!python -m spacy download en_core_web_lg

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
import nltk
import spacy

In [ ]:
# nltk.download('stopwords')
# tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
# tokenizer.tokenize("I'm studying NLP and it's interesting.")
# stop_words = nltk.corpus.stopwords.words("english")
# print(stop_words[:20])
# print(len(stop_words))

In [ ]:
try:
    tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
    tokenizer.tokenize("I'm studying NLP and it's interesting.")
    stop_words = nltk.corpus.stopwords.words("english")
    nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"]) # disable=["parser", "tagger", "ner"]
except Exception:
    nltk.download("stopwords")
    nltk.download("punkt")
    spacy.cli.download("en_core_web_lg")
    tokenizer = nltk.tokenize.RegexpTokenizer(r"\w+'?\w+|\w+'")
    tokenizer.tokenize("I'm studying NLP and it's interesting.")
    stop_words = nltk.corpus.stopwords.words("english")
    nlp = spacy.load("en_core_web_lg", disable=["parser", "ner"])  # disable=["parser", "tagger", "ner"]
    print(stop_words[:20])
    print(len(stop_words))

In [ ]:
# Create our corpus for training and perform some classic NLP preprocessing
dataset = pd.read_csv("/content/twitter.csv")

In [ ]:
text_data = list(
    map(lambda x: tokenizer.tokenize(x.lower()), dataset['text'])
)
text_data = [
    [token.lemma_ for word in text for token in nlp(word)] for text in text_data
]
label_data = list(map(lambda x: x, dataset["feeling"]))
assert len(text_data) == len(label_data), f"{len(text_data)} does not equal {len(label_data)}"

In [ ]:
EMBEDDING_DIM = 100
model = Word2Vec(
    text_data, vector_size=EMBEDDING_DIM, window=5, min_count=1, workers=4
)
word_vectors = model.wv
print(f"Vocabulary Length: {len(model.wv)}")
del model

In [ ]:
padding_value = len(word_vectors.index_to_key)
# Embeddings are needed to give semantic value to the inputs of an LSTM
embedding_weights = torch.Tensor(word_vectors.vectors)

In [ ]:
# RNN model
class RNN(nn.Module):
    def __init__(
            self,
            input_dim,
            embedding_dim,
            hidden_dim,
            output_dim,
            embedding_weights
    ):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_weights)
        self.rnn = nn.RNN(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, text_lengths):
        embedded = self.embedding(x)
        packed_embedded = nn.utils.rnn.pad_packed_sequence(embedded, text_lengths)
        packed_output, hidden = self.rnn(packed_embedded)
        output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output)

        return self.fc(hidden.squeeze(0))

In [ ]:
INPUT_DIM = padding_value
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1

In [ ]:
rnn_model = RNN(
    INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, embedding_weights
)

In [ ]:
rnn_optimizer = optim.SGD(rnn_model.parameters(), lr=1e-3)
rnn_criterion = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# LSTM model
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        pass

    def forward(self, x):
        pass